# Module 7 | Class 5 -- YOLO Object Detection

**Objective:** Use a pre-trained YOLOv8 model to detect objects in images, visualize the results, and understand detection outputs (bounding boxes, classes, confidence scores).

YOLO (You Only Look Once) is a real-time object detection model. It processes the entire image in a single forward pass, making it fast enough for video streams.

## 1. Setup

In [ ]:
!pip install -q ultralytics Pillow requests

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from ultralytics import YOLO
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import numpy as np

print("Setup complete.")

## 2. Load Pre-trained YOLOv8 Model

YOLOv8n is the "nano" variant -- smallest and fastest. It is pre-trained on the COCO dataset (80 object classes: person, car, dog, chair, etc.).

In [ ]:
model = YOLO('yolov8n.pt')
print(f"Model loaded: {model.model_name if hasattr(model, 'model_name') else 'yolov8n'}")
print(f"Number of classes: {len(model.names)}")
print(f"Sample classes: {dict(list(model.names.items())[:15])}")

## 3. Helper Functions

In [ ]:
def load_image_from_url(url):
    """Download an image from a URL and return as PIL Image."""
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert('RGB')


def detect_and_display(image_url, title="Detection Result"):
    """Run YOLO on an image URL, display annotated result, and print detections."""
    # Load image
    img = load_image_from_url(image_url)
    print(f"Image size: {img.size}")

    # Run detection
    results = model(img, verbose=False)
    result = results[0]

    # Print detections
    boxes = result.boxes
    print(f"\nObjects detected: {len(boxes)}")
    print(f"{'Class':<20s} {'Confidence':>10s} {'Bounding Box (x1, y1, x2, y2)'}")
    print("-" * 65)
    for box in boxes:
        cls_id = int(box.cls[0])
        cls_name = model.names[cls_id]
        conf = float(box.conf[0])
        coords = box.xyxy[0].tolist()
        coords_str = f"({coords[0]:.0f}, {coords[1]:.0f}, {coords[2]:.0f}, {coords[3]:.0f})"
        print(f"{cls_name:<20s} {conf:>10.4f} {coords_str}")

    # Display annotated image
    annotated = result.plot()
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].imshow(np.array(img))
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(annotated)
    axes[1].set_title(title)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    return result

print("Helper functions ready.")

## 4. Run Detection on Sample Images

We use publicly available images from Unsplash and similar sources.

In [ ]:
# Image 1: Street scene with people and vehicles
print("=" * 60)
print("IMAGE 1: Street Scene")
print("=" * 60)
r1 = detect_and_display(
    "https://ultralytics.com/images/bus.jpg",
    title="Street Scene Detections"
)

In [ ]:
# Image 2: Another scene
print("=" * 60)
print("IMAGE 2: Zidane")
print("=" * 60)
r2 = detect_and_display(
    "https://ultralytics.com/images/zidane.jpg",
    title="People Detection"
)

In [ ]:
# Image 3: Indoor/office scene
print("=" * 60)
print("IMAGE 3: Dog on Beach")
print("=" * 60)
r3 = detect_and_display(
    "https://images.unsplash.com/photo-1558788353-f76d92427f16?w=640",
    title="Beach Scene Detections"
)

In [ ]:
# Image 4: Traffic
print("=" * 60)
print("IMAGE 4: Traffic")
print("=" * 60)
r4 = detect_and_display(
    "https://images.unsplash.com/photo-1449824913935-59a10b8d2000?w=640",
    title="Traffic Detections"
)

## 5. Understanding the Output

Each detection gives you:
- **Class:** What object was detected (from the 80 COCO classes)
- **Confidence:** How certain the model is (0.0 to 1.0)
- **Bounding box:** The rectangle coordinates (x1, y1, x2, y2) defining where the object is

Let's look at detection statistics for one of our results.

In [ ]:
# Summarize detections from the first image
from collections import Counter

def summarize_detections(result):
    """Print a summary of detected object types and confidence stats."""
    boxes = result.boxes
    if len(boxes) == 0:
        print("No objects detected.")
        return

    classes = [model.names[int(c)] for c in boxes.cls]
    confs = [float(c) for c in boxes.conf]

    print(f"Total detections: {len(boxes)}")
    print(f"\nObject counts:")
    for cls, count in Counter(classes).most_common():
        print(f"  {cls}: {count}")

    print(f"\nConfidence stats:")
    print(f"  Min: {min(confs):.4f}")
    print(f"  Max: {max(confs):.4f}")
    print(f"  Mean: {np.mean(confs):.4f}")

summarize_detections(r1)

## 6. Adjusting Confidence Threshold

By default, YOLO filters out detections below a confidence threshold. You can adjust this.

In [ ]:
img = load_image_from_url("https://ultralytics.com/images/bus.jpg")

for conf_thresh in [0.1, 0.25, 0.5, 0.7]:
    results = model(img, conf=conf_thresh, verbose=False)
    n_detections = len(results[0].boxes)
    print(f"Confidence threshold = {conf_thresh:.2f} -> {n_detections} detections")

Lower threshold = more detections (but more false positives).  
Higher threshold = fewer detections (but higher precision).

This is the precision-recall tradeoff in action.

---
## TODO: Student Exercise

**Task:** Find 5 images online and run YOLO detection on them.

Requirements:
1. Use publicly accessible image URLs (Unsplash, Pexels, Wikipedia Commons, etc.)
2. Try to include a variety: indoor, outdoor, animals, vehicles, people
3. For each image:
   - Run `detect_and_display()`
   - Document what was detected vs what you can actually see
   - Note any missed objects or incorrect detections
4. Write a short summary: What types of objects does YOLOv8 handle well? Where does it struggle?

In [ ]:
# TODO: Image 1
# url_1 = "your_url_here"
# detect_and_display(url_1, title="My Image 1")
# Observation: ...


In [ ]:
# TODO: Image 2


In [ ]:
# TODO: Image 3


In [ ]:
# TODO: Image 4


In [ ]:
# TODO: Image 5


In [ ]:
# TODO: Summary
# What types of objects does YOLOv8 handle well?
# Where does it struggle?
# (Write as comments or print statements)
